<a href="https://colab.research.google.com/github/esila-keskin/Physx_sim/blob/main/notebook3e7d0a7052.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install warp-lang -q
!pip install opencv-python -q
print("Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 MB 7.2 MB/s eta 0:00:00
Packages installed


In [3]:
import warp as wp
import numpy as np
import os, cv2
import pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
wp.init()
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

Warp 1.11.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.1
Torch: 2.10.0+cu128 | CUDA: True
  GPU 0: Tesla T4


In [4]:
#  num_workers=0 prevents the torch._dynamo multiprocessing crash
class PhysicsCNN(nn.Module):
    def __init__(self, in_ch, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,   64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,  128,  3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(128*8*8, 256), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, out_dim)
        )
    def forward(self, x): return self.fc(self.conv(x))

def train_and_eval(model, train_ds, val_ds, epochs=20, name=''):
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            criterion(model(x.cuda()), y.cuda()).backward()
            optimizer.step()
        model.eval()
        vl = np.mean([criterion(model(x.cuda()), y.cuda()).item() for x, y in val_loader])
        scheduler.step(vl)
        if (epoch+1) % 5 == 0:
            print(f"  {name} Epoch {epoch+1}  val_loss={vl:.4f}")
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, y in val_loader:
            all_p.append(model(x.cuda()).cpu())
            all_t.append(y)
    return torch.cat(all_p).numpy(), torch.cat(all_t).numpy()

def get_splits(df):
    n  = int(0.8 * df['step'].nunique())
    ts = df['step'].unique()[:n]
    vs = df['step'].unique()[n:]
    return df[df['step'].isin(ts)].reset_index(drop=True), df[df['step'].isin(vs)].reset_index(drop=True)

print("Model and helpers ready")

Model and helpers ready


In [5]:
# Fixed radius r=0.05, sliding friction
dt = 0.01
steps = 300
num_envs = 100
balls_per_env = 4
image_size    = 64
radius = 0.05
total_balls   = num_envs * balls_per_env

os.makedirs("dataset_v4/images", exist_ok=True)
os.makedirs("dataset_v4/masks",  exist_ok=True)

pos_np = np.random.uniform(0.2, 0.8, (total_balls, 2)).astype(np.float32)
vel_np = np.random.uniform(-1.5, 1.5, (total_balls, 2)).astype(np.float32)

position = wp.array(pos_np, dtype=wp.vec2, device="cuda")
velocity = wp.array(vel_np, dtype=wp.vec2, device="cuda")

gravity_per_env     = np.random.uniform(-12.0, -5.0, num_envs).astype(np.float32)
restitution_per_env = np.random.uniform(0.7,  0.95,  num_envs).astype(np.float32)
friction_per_env    = np.random.uniform(0.0,  1.0,   num_envs).astype(np.float32)

gravity_wp     = wp.array(gravity_per_env,     dtype=float, device="cuda")
restitution_wp = wp.array(restitution_per_env, dtype=float, device="cuda")
friction_wp    = wp.array(friction_per_env,    dtype=float, device="cuda")

@wp.kernel
def simulate_v4(
    pos: wp.array(dtype=wp.vec2), vel: wp.array(dtype=wp.vec2),
    gravity_env: wp.array(dtype=float), restitution_env: wp.array(dtype=float),
    friction_env: wp.array(dtype=float), dt: float, r: float,
    balls_per_env: int, total_balls: int):
    i      = wp.tid()
    env_id = i // balls_per_env
    g      = gravity_env[env_id]
    rest   = restitution_env[env_id]
    mu     = friction_env[env_id]
    vel[i] = wp.vec2(vel[i][0], vel[i][1] + g * dt)
    pos[i] = pos[i] + vel[i] * dt
    if pos[i][1] < r:
        pos[i] = wp.vec2(pos[i][0], r)
        vel[i] = wp.vec2(vel[i][0], -vel[i][1] * rest)
        fi = mu * wp.abs(vel[i][1])
        if vel[i][0] > 0.0: vel[i] = wp.vec2(wp.max(0.0, vel[i][0] - fi), vel[i][1])
        else:                vel[i] = wp.vec2(wp.min(0.0, vel[i][0] + fi), vel[i][1])
    if pos[i][1] > 1.0 - r:
        pos[i] = wp.vec2(pos[i][0], 1.0 - r)
        vel[i] = wp.vec2(vel[i][0], -vel[i][1] * rest)
        fi = mu * wp.abs(vel[i][1])
        if vel[i][0] > 0.0: vel[i] = wp.vec2(wp.max(0.0, vel[i][0] - fi), vel[i][1])
        else:                vel[i] = wp.vec2(wp.min(0.0, vel[i][0] + fi), vel[i][1])
    if pos[i][0] < r:
        pos[i] = wp.vec2(r, pos[i][1])
        vel[i] = wp.vec2(-vel[i][0] * rest, vel[i][1])
        fi = mu * wp.abs(vel[i][0])
        if vel[i][1] > 0.0: vel[i] = wp.vec2(vel[i][0], wp.max(0.0, vel[i][1] - fi))
        else:                vel[i] = wp.vec2(vel[i][0], wp.min(0.0, vel[i][1] + fi))
    if pos[i][0] > 1.0 - r:
        pos[i] = wp.vec2(1.0 - r, pos[i][1])
        vel[i] = wp.vec2(-vel[i][0] * rest, vel[i][1])
        fi = mu * wp.abs(vel[i][0])
        if vel[i][1] > 0.0: vel[i] = wp.vec2(vel[i][0], wp.max(0.0, vel[i][1] - fi))
        else:                vel[i] = wp.vec2(vel[i][0], wp.min(0.0, vel[i][1] + fi))
    for j in range(total_balls):
        if j == i: continue
        if j // balls_per_env != env_id: continue
        delta = pos[i] - pos[j]
        dist  = wp.length(delta)
        if dist < 2.0 * r and dist > 1e-6:
            normal = delta / dist
            pos[i] = pos[i] + normal * (2.0 * r - dist) * 0.5
            rel_vel = vel[i] - vel[j]
            vn = wp.dot(rel_vel, normal)
            if vn < 0.0:
                impulse = -(1.0 + rest) * vn / 2.0
                vel[i]  = vel[i] + normal * impulse
                tangent = wp.vec2(-normal[1], normal[0])
                vt = wp.dot(rel_vel, tangent)
                fi = mu * wp.abs(impulse)
                if vt > 0.0: tc = wp.min(vt,  fi)
                else:        tc = wp.max(vt, -fi)
                vel[i] = vel[i] - tangent * tc * 0.5

colors = [(255,0,0),(0,255,0),(0,0,255),(255,255,0)]
labels = []
print("Generating dataset_v4...")
for step in range(steps):
    wp.launch(simulate_v4, dim=total_balls,
              inputs=[position, velocity, gravity_wp, restitution_wp,
                      friction_wp, dt, radius, balls_per_env, total_balls],
              device="cuda")
    pos_cpu = position.numpy()
    vel_cpu = velocity.numpy()
    for env in range(num_envs):
        img  = np.zeros((image_size, image_size, 3), dtype=np.uint8)
        mask = np.zeros((image_size, image_size),    dtype=np.uint8)
        for b in range(balls_per_env):
            idx = env * balls_per_env + b
            x, y = pos_cpu[idx]
            px = int(x * image_size)
            py = int((1.0 - y) * image_size)
            rad_pix = int(radius * image_size)
            cv2.circle(img,  (px, py), rad_pix, colors[b], -1)
            cv2.circle(mask, (px, py), rad_pix, b+1,       -1)
            labels.append([step, env, b, x, y,
                           vel_cpu[idx][0], vel_cpu[idx][1],
                           gravity_per_env[env], restitution_per_env[env],
                           friction_per_env[env],
                           px-rad_pix, py-rad_pix, px+rad_pix, py+rad_pix])
        cv2.imwrite(f"dataset_v4/images/env{env}_step{step}.png", img)
        cv2.imwrite(f"dataset_v4/masks/env{env}_step{step}.png",  mask)
    if step % 50 == 0: print(f" Step {step}/{steps}")

df_v4 = pd.DataFrame(labels, columns=[
    "step","env_id","object_id","x","y","vx","vy",
    "gravity","restitution","friction",
    "x_min","y_min","x_max","y_max"])
df_v4.to_csv("dataset_v4/labels.csv", index=False)
print(f"Done - {len(df_v4)} annotations, {steps*num_envs} images")

Generating dataset_v4...
Module __main__ 300ed85 load on device 'cuda:0' took 1863.17 ms  (compiled)
 Step 0/300
 Step 50/300
 Step 100/300
 Step 150/300
 Step 200/300
 Step 250/300
Done - 120000 annotations, 30000 images


In [6]:
# Variable radii r in [0.03,0.08], sliding friction
dt = 0.01
steps = 300
num_envs = 100
balls_per_env = 4
image_size    = 64
total_balls   = num_envs * balls_per_env

os.makedirs("dataset_v5/images", exist_ok=True)
os.makedirs("dataset_v5/masks",  exist_ok=True)

radii_np = np.random.uniform(0.03, 0.08, total_balls).astype(np.float32)
radii_wp = wp.array(radii_np, dtype=float, device="cuda")

pos_np = np.random.uniform(0.2, 0.8, (total_balls, 2)).astype(np.float32)
vel_np = np.random.uniform(-1.5, 1.5, (total_balls, 2)).astype(np.float32)
position = wp.array(pos_np, dtype=wp.vec2, device="cuda")
velocity = wp.array(vel_np, dtype=wp.vec2, device="cuda")

gravity_per_env     = np.random.uniform(-12.0, -5.0, num_envs).astype(np.float32)
restitution_per_env = np.random.uniform(0.70,  0.95, num_envs).astype(np.float32)
friction_per_env    = np.random.uniform(0.0,   1.0,  num_envs).astype(np.float32)
gravity_wp     = wp.array(gravity_per_env,     dtype=float, device="cuda")
restitution_wp = wp.array(restitution_per_env, dtype=float, device="cuda")
friction_wp    = wp.array(friction_per_env,    dtype=float, device="cuda")

@wp.kernel
def simulate_v5(
    pos: wp.array(dtype=wp.vec2), vel: wp.array(dtype=wp.vec2),
    radii: wp.array(dtype=float),
    gravity_env: wp.array(dtype=float), restitution_env: wp.array(dtype=float),
    friction_env: wp.array(dtype=float), dt: float,
    balls_per_env: int, total_balls: int):
    i      = wp.tid()
    env_id = i // balls_per_env
    g      = gravity_env[env_id]
    rest   = restitution_env[env_id]
    mu     = friction_env[env_id]
    r      = radii[i]
    vel[i] = wp.vec2(vel[i][0], vel[i][1] + g * dt)
    pos[i] = pos[i] + vel[i] * dt
    if pos[i][1] < r:
        pos[i] = wp.vec2(pos[i][0], r)
        vel[i] = wp.vec2(vel[i][0], -vel[i][1] * rest)
        fi = mu * wp.abs(vel[i][1])
        if vel[i][0] > 0.0: vel[i] = wp.vec2(wp.max(0.0, vel[i][0] - fi), vel[i][1])
        else:                vel[i] = wp.vec2(wp.min(0.0, vel[i][0] + fi), vel[i][1])
    if pos[i][1] > 1.0 - r:
        pos[i] = wp.vec2(pos[i][0], 1.0 - r)
        vel[i] = wp.vec2(vel[i][0], -vel[i][1] * rest)
        fi = mu * wp.abs(vel[i][1])
        if vel[i][0] > 0.0: vel[i] = wp.vec2(wp.max(0.0, vel[i][0] - fi), vel[i][1])
        else:                vel[i] = wp.vec2(wp.min(0.0, vel[i][0] + fi), vel[i][1])
    if pos[i][0] < r:
        pos[i] = wp.vec2(r, pos[i][1])
        vel[i] = wp.vec2(-vel[i][0] * rest, vel[i][1])
        fi = mu * wp.abs(vel[i][0])
        if vel[i][1] > 0.0: vel[i] = wp.vec2(vel[i][0], wp.max(0.0, vel[i][1] - fi))
        else:                vel[i] = wp.vec2(vel[i][0], wp.min(0.0, vel[i][1] + fi))
    if pos[i][0] > 1.0 - r:
        pos[i] = wp.vec2(1.0 - r, pos[i][1])
        vel[i] = wp.vec2(-vel[i][0] * rest, vel[i][1])
        fi = mu * wp.abs(vel[i][0])
        if vel[i][1] > 0.0: vel[i] = wp.vec2(vel[i][0], wp.max(0.0, vel[i][1] - fi))
        else:                vel[i] = wp.vec2(vel[i][0], wp.min(0.0, vel[i][1] + fi))
    for j in range(total_balls):
        if j == i: continue
        if j // balls_per_env != env_id: continue
        r_j   = radii[j]
        delta = pos[i] - pos[j]
        dist  = wp.length(delta)
        if dist < r + r_j and dist > 1e-6:
            normal = delta / dist
            pos[i] = pos[i] + normal * (r + r_j - dist) * 0.5
            rel_vel = vel[i] - vel[j]
            vn = wp.dot(rel_vel, normal)
            if vn < 0.0:
                impulse = -(1.0 + rest) * vn / 2.0
                vel[i]  = vel[i] + normal * impulse
                tangent = wp.vec2(-normal[1], normal[0])
                vt = wp.dot(rel_vel, tangent)
                fi = mu * wp.abs(impulse)
                if vt > 0.0: tc = wp.min(vt,  fi)
                else:        tc = wp.max(vt, -fi)
                vel[i] = vel[i] - tangent * tc * 0.5

colors = [(255,0,0),(0,255,0),(0,0,255),(255,255,0)]
labels = []
print("Generating dataset_v5...")
for step in range(steps):
    wp.launch(simulate_v5, dim=total_balls,
              inputs=[position, velocity, radii_wp,
                      gravity_wp, restitution_wp, friction_wp,
                      dt, balls_per_env, total_balls],
              device="cuda")
    pos_cpu = position.numpy()
    vel_cpu = velocity.numpy()
    for env in range(num_envs):
        img  = np.zeros((image_size, image_size, 3), dtype=np.uint8)
        mask = np.zeros((image_size, image_size),    dtype=np.uint8)
        for b in range(balls_per_env):
            idx     = env * balls_per_env + b
            x, y    = pos_cpu[idx]
            px      = int(x * image_size)
            py      = int((1.0 - y) * image_size)
            rad_pix = max(1, int(radii_np[idx] * image_size))
            cv2.circle(img,  (px, py), rad_pix, colors[b], -1)
            cv2.circle(mask, (px, py), rad_pix, b+1,       -1)
            labels.append([step, env, b, x, y,
                           vel_cpu[idx][0], vel_cpu[idx][1],
                           float(radii_np[idx]),
                           gravity_per_env[env], restitution_per_env[env],
                           friction_per_env[env],
                           px-rad_pix, py-rad_pix, px+rad_pix, py+rad_pix])
        cv2.imwrite(f"dataset_v5/images/env{env}_step{step}.png", img)
        cv2.imwrite(f"dataset_v5/masks/env{env}_step{step}.png",  mask)
    if step % 50 == 0: print(f"  Step {step}/{steps}")

df_v5 = pd.DataFrame(labels, columns=[
    "step","env_id","object_id","x","y","vx","vy","radius",
    "gravity","restitution","friction",
    "x_min","y_min","x_max","y_max"])
df_v5.to_csv("dataset_v5/labels.csv", index=False)
print(f"Done v5 - {len(df_v5)} annotations")
print(f"Radius range: {df_v5['radius'].min():.3f} to {df_v5['radius'].max():.3f}")

Generating dataset_v5...
Module __main__ b111f76 load on device 'cuda:0' took 2609.29 ms  (compiled)
  Step 0/300
  Step 50/300
  Step 100/300
  Step 150/300
  Step 200/300
  Step 250/300
Done v5 - 120000 annotations
Radius range: 0.030 to 0.080


In [7]:
# Variable radii + rolling friction + angular velocity omega
dt = 0.01
steps = 300
num_envs = 100
balls_per_env = 4
image_size = 64
total_balls   = num_envs * balls_per_env
OMEGA_MAX = 30.0

os.makedirs("dataset_v6/images", exist_ok=True)
os.makedirs("dataset_v6/masks",  exist_ok=True)

radii_np   = np.random.uniform(0.03, 0.08, total_balls).astype(np.float32)
inertia_np = (0.4 * radii_np * radii_np).astype(np.float32)
radii_wp   = wp.array(radii_np,   dtype=float, device="cuda")
inertia_wp = wp.array(inertia_np, dtype=float, device="cuda")

pos_np   = np.random.uniform(0.2, 0.8, (total_balls, 2)).astype(np.float32)
vel_np   = np.random.uniform(-1.5, 1.5, (total_balls, 2)).astype(np.float32)
omega_np = np.zeros(total_balls, dtype=np.float32)
position = wp.array(pos_np,   dtype=wp.vec2, device="cuda")
velocity = wp.array(vel_np,   dtype=wp.vec2, device="cuda")
omega    = wp.array(omega_np, dtype=float,   device="cuda")

gravity_per_env     = np.random.uniform(-12.0, -5.0, num_envs).astype(np.float32)
restitution_per_env = np.random.uniform(0.70,  0.95, num_envs).astype(np.float32)
friction_per_env    = np.random.uniform(0.0,   1.0,  num_envs).astype(np.float32)
gravity_wp     = wp.array(gravity_per_env,     dtype=float, device="cuda")
restitution_wp = wp.array(restitution_per_env, dtype=float, device="cuda")
friction_wp    = wp.array(friction_per_env,    dtype=float, device="cuda")

@wp.kernel
def simulate_v6(
    pos:             wp.array(dtype=wp.vec2),
    vel:             wp.array(dtype=wp.vec2),
    omega:           wp.array(dtype=float),
    radii:           wp.array(dtype=float),
    inertia:         wp.array(dtype=float),
    gravity_env:     wp.array(dtype=float),
    restitution_env: wp.array(dtype=float),
    friction_env:    wp.array(dtype=float),
    dt:              float,
    omega_max:       float,
    balls_per_env:   int,
    total_balls:     int,
):
    i      = wp.tid()
    env_id = i // balls_per_env
    g      = gravity_env[env_id]
    rest   = restitution_env[env_id]
    mu     = friction_env[env_id]
    r      = radii[i]
    I      = inertia[i]

    vel[i] = wp.vec2(vel[i][0], vel[i][1] + g * dt)
    pos[i] = pos[i] + vel[i] * dt

    if pos[i][1] < r:
        pos[i]  = wp.vec2(pos[i][0], r)
        J_n     = -(1.0 + rest) * vel[i][1]
        vel[i]  = wp.vec2(vel[i][0], vel[i][1] + J_n)
        slip    = vel[i][0] - omega[i] * r
        J_t     = wp.clamp(-slip * 0.5, -mu * wp.abs(J_n), mu * wp.abs(J_n))
        vel[i]  = wp.vec2(vel[i][0] + J_t, vel[i][1])
        omega[i] = wp.clamp(omega[i] - J_t * r / I, -omega_max, omega_max)

    if pos[i][1] > 1.0 - r:
        pos[i]  = wp.vec2(pos[i][0], 1.0 - r)
        J_n     = -(1.0 + rest) * vel[i][1]
        vel[i]  = wp.vec2(vel[i][0], vel[i][1] + J_n)
        slip    = vel[i][0] + omega[i] * r
        J_t     = wp.clamp(-slip * 0.5, -mu * wp.abs(J_n), mu * wp.abs(J_n))
        vel[i]  = wp.vec2(vel[i][0] + J_t, vel[i][1])
        omega[i] = wp.clamp(omega[i] + J_t * r / I, -omega_max, omega_max)

    if pos[i][0] < r:
        pos[i]  = wp.vec2(r, pos[i][1])
        J_n     = -(1.0 + rest) * vel[i][0]
        vel[i]  = wp.vec2(vel[i][0] + J_n, vel[i][1])
        slip    = vel[i][1] + omega[i] * r
        J_t     = wp.clamp(-slip * 0.5, -mu * wp.abs(J_n), mu * wp.abs(J_n))
        vel[i]  = wp.vec2(vel[i][0], vel[i][1] + J_t)
        omega[i] = wp.clamp(omega[i] - J_t * r / I, -omega_max, omega_max)

    if pos[i][0] > 1.0 - r:
        pos[i]  = wp.vec2(1.0 - r, pos[i][1])
        J_n     = -(1.0 + rest) * vel[i][0]
        vel[i]  = wp.vec2(vel[i][0] + J_n, vel[i][1])
        slip    = vel[i][1] - omega[i] * r
        J_t     = wp.clamp(-slip * 0.5, -mu * wp.abs(J_n), mu * wp.abs(J_n))
        vel[i]  = wp.vec2(vel[i][0], vel[i][1] + J_t)
        omega[i] = wp.clamp(omega[i] + J_t * r / I, -omega_max, omega_max)

    for j in range(total_balls):
        if j == i: continue
        if j // balls_per_env != env_id: continue
        r_j   = radii[j]
        delta = pos[i] - pos[j]
        dist  = wp.length(delta)
        if dist < r + r_j and dist > 1e-6:
            normal = delta / dist
            pos[i] = pos[i] + normal * (r + r_j - dist) * 0.5
            rel_vel = vel[i] - vel[j]
            vn = wp.dot(rel_vel, normal)
            if vn < 0.0:
                J_n     = -(1.0 + rest) * vn / 2.0
                vel[i]  = vel[i] + normal * J_n
                tangent = wp.vec2(-normal[1], normal[0])
                vt      = wp.dot(rel_vel, tangent)
                slip_c  = vt - (omega[i] + omega[j]) * r
                J_t     = wp.clamp(-slip_c * 0.25, -mu * wp.abs(J_n), mu * wp.abs(J_n))
                vel[i]  = vel[i] - tangent * J_t * 0.5
                omega[i] = wp.clamp(omega[i] + J_t * r / I, -omega_max, omega_max)

colors = [(255,0,0),(0,255,0),(0,0,255),(255,255,0)]
labels = []
print("Generating dataset_v6 - variable radii + rolling friction...")
for step in range(steps):
    wp.launch(simulate_v6, dim=total_balls,
              inputs=[position, velocity, omega, radii_wp, inertia_wp,
                      gravity_wp, restitution_wp, friction_wp,
                      dt, OMEGA_MAX, balls_per_env, total_balls],
              device="cuda")
    pos_cpu   = position.numpy()
    vel_cpu   = velocity.numpy()
    omega_cpu = omega.numpy()
    for env in range(num_envs):
        img  = np.zeros((image_size, image_size, 3), dtype=np.uint8)
        mask = np.zeros((image_size, image_size),    dtype=np.uint8)
        for b in range(balls_per_env):
            idx     = env * balls_per_env + b
            x, y    = pos_cpu[idx]
            px      = int(x * image_size)
            py      = int((1.0 - y) * image_size)
            rad_pix = max(1, int(radii_np[idx] * image_size))
            cv2.circle(img,  (px, py), rad_pix, colors[b], -1)
            cv2.circle(mask, (px, py), rad_pix, b+1,       -1)
            theta = float(omega_cpu[idx]) * step * dt
            sx = int(px + rad_pix * 0.8 * np.cos(theta))
            sy = int(py + rad_pix * 0.8 * np.sin(theta))
            cv2.line(img, (px, py), (sx, sy), (255,255,255), 1)
            labels.append([step, env, b, x, y,
                           vel_cpu[idx][0], vel_cpu[idx][1],
                           float(omega_cpu[idx]),
                           float(radii_np[idx]),
                           gravity_per_env[env], restitution_per_env[env],
                           friction_per_env[env],
                           px-rad_pix, py-rad_pix, px+rad_pix, py+rad_pix])
        cv2.imwrite(f"dataset_v6/images/env{env}_step{step}.png", img)
        cv2.imwrite(f"dataset_v6/masks/env{env}_step{step}.png",  mask)
    if step % 50 == 0: print(f"  Step {step}/{steps}")

df_v6 = pd.DataFrame(labels, columns=[
    "step","env_id","object_id","x","y","vx","vy","omega","radius",
    "gravity","restitution","friction",
    "x_min","y_min","x_max","y_max"])
df_v6.to_csv("dataset_v6/labels.csv", index=False)
print(f"Done v6 - {len(df_v6)} annotations")
print(f"Radius range: {df_v6['radius'].min():.3f} to {df_v6['radius'].max():.3f}")
print(f"Omega range:  {df_v6['omega'].min():.2f} to {df_v6['omega'].max():.2f}  (must be within -30 to +30)")

Generating dataset_v6 - variable radii + rolling friction...
Module __main__ d681c5a load on device 'cuda:0' took 3774.13 ms  (compiled)
  Step 0/300
  Step 50/300
  Step 100/300
  Step 150/300
  Step 200/300
  Step 250/300
Done v6 - 120000 annotations
Radius range: 0.030 to 0.080
Omega range:  -30.00 to 30.00  (must be within -30 to +30)


In [ ]:
# All 5 dataset classes
class SingleFrameDS(Dataset):
    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.t = transforms.Compose([transforms.ToTensor(),
                 transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
        self.vx_mean, self.vx_std = df['vx'].mean(), df['vx'].std()+1e-8
        self.vy_mean, self.vy_std = df['vy'].mean(), df['vy'].std()+1e-8
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.t(Image.open(f"{self.img_dir}/env{int(row.env_id)}_step{int(row.step)}.png").convert("RGB"))
        return img, torch.tensor([(row.vx-self.vx_mean)/self.vx_std,
                                   (row.vy-self.vy_mean)/self.vy_std], dtype=torch.float32)

class TwoFrameDS(Dataset):
    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.valid = [i for i in range(len(df)) if int(df.iloc[i].step) > 0]
        self.t = transforms.Compose([transforms.ToTensor(),
                 transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
        self.vx_mean, self.vx_std = df['vx'].mean(), df['vx'].std()+1e-8
        self.vy_mean, self.vy_std = df['vy'].mean(), df['vy'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        curr = self.t(Image.open(f"{self.img_dir}/env{env}_step{step}.png").convert("RGB"))
        prev = self.t(Image.open(f"{self.img_dir}/env{env}_step{step-1}.png").convert("RGB"))
        return torch.cat([curr,prev],dim=0), torch.tensor([(row.vx-self.vx_mean)/self.vx_std,
                                                            (row.vy-self.vy_mean)/self.vy_std], dtype=torch.float32)

class EventDS(Dataset):
    def __init__(self, df, img_dir, threshold=10):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.threshold = threshold
        self.valid = [i for i in range(len(df)) if int(df.iloc[i].step) > 0]
        self.vx_mean, self.vx_std = df['vx'].mean(), df['vx'].std()+1e-8
        self.vy_mean, self.vy_std = df['vy'].mean(), df['vy'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        curr = np.array(Image.open(f"{self.img_dir}/env{env}_step{step}.png")).astype(float)
        prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{step-1}.png")).astype(float)
        diff = curr.mean(2) - prev.mean(2)
        event = torch.tensor(np.stack([(diff>self.threshold).astype(np.float32),
                                        (diff<-self.threshold).astype(np.float32)]))
        return event, torch.tensor([(row.vx-self.vx_mean)/self.vx_std,
                                     (row.vy-self.vy_mean)/self.vy_std], dtype=torch.float32)

class SeqEventDS(Dataset):
    # Used for v4 and v5 — predicts vx, vy, gravity, restitution, friction (5 outputs)
    def __init__(self, df, img_dir, seq_len=5, threshold=10):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.seq_len = seq_len
        self.threshold = threshold
        self.valid = [i for i in range(len(df)) if int(df.iloc[i].step) >= seq_len]
        self.vx_mean, self.vx_std = df['vx'].mean(), df['vx'].std()+1e-8
        self.vy_mean, self.vy_std = df['vy'].mean(), df['vy'].std()+1e-8
        self.g_mean,  self.g_std  = df['gravity'].mean(), df['gravity'].std()+1e-8
        self.r_mean,  self.r_std  = df['restitution'].mean(), df['restitution'].std()+1e-8
        self.f_mean,  self.f_std  = df['friction'].mean(), df['friction'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        frames = []
        for s in range(step-self.seq_len+1, step+1):
            img = np.array(Image.open(f"{self.img_dir}/env{env}_step{s}.png")).astype(float)
            if s > step-self.seq_len+1:
                prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{s-1}.png")).astype(float)
                diff = img.mean(2) - prev.mean(2)
                frames += [(diff>self.threshold).astype(np.float32),
                            (diff<-self.threshold).astype(np.float32)]
            else:
                frames += [np.zeros((64,64),dtype=np.float32),
                            np.zeros((64,64),dtype=np.float32)]
        return torch.tensor(np.stack(frames)), torch.tensor([
            (row.vx-self.vx_mean)/self.vx_std,
            (row.vy-self.vy_mean)/self.vy_std,
            (row.gravity-self.g_mean)/self.g_std,
            (row.restitution-self.r_mean)/self.r_std,
            (row.friction-self.f_mean)/self.f_std], dtype=torch.float32)

print("Dataset classes for v4/v5 ready")

class SeqEventV6DS(Dataset):
    def __init__(self, df, img_dir, seq_len=5, threshold=10):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.seq_len = seq_len
        self.threshold = threshold
        self.valid = [i for i in range(len(df)) if int(df.iloc[i].step) >= seq_len]
        self.vx_mean,  self.vx_std  = df['vx'].mean(),          df['vx'].std()+1e-8
        self.vy_mean,  self.vy_std  = df['vy'].mean(),          df['vy'].std()+1e-8
        self.g_mean,   self.g_std   = df['gravity'].mean(),     df['gravity'].std()+1e-8
        self.r_mean,   self.r_std   = df['restitution'].mean(), df['restitution'].std()+1e-8
        self.f_mean,   self.f_std   = df['friction'].mean(),    df['friction'].std()+1e-8
        self.om_mean,  self.om_std  = df['omega'].mean(),       df['omega'].std()+1e-8
        self.rad_mean, self.rad_std = df['radius'].mean(),      df['radius'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        frames = []
        for s in range(step-self.seq_len+1, step+1):
            img = np.array(Image.open(f"{self.img_dir}/env{env}_step{s}.png")).astype(float)
            if s > step-self.seq_len+1:
                prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{s-1}.png")).astype(float)
                diff = img.mean(2) - prev.mean(2)
                frames += [(diff>self.threshold).astype(np.float32),
                            (diff<-self.threshold).astype(np.float32)]
            else:
                frames += [np.zeros((64,64),dtype=np.float32),
                            np.zeros((64,64),dtype=np.float32)]
        return torch.tensor(np.stack(frames)), torch.tensor([
            (row.vx - self.vx_mean)  / self.vx_std,
            (row.vy - self.vy_mean)  / self.vy_std,
            (row.gravity     - self.g_mean)   / self.g_std,
            (row.restitution - self.r_mean)   / self.r_std,
            (row.friction    - self.f_mean)   / self.f_std,
            (row.omega       - self.om_mean)  / self.om_std,
            (row.radius      - self.rad_mean) / self.rad_std], dtype=torch.float32)

print("Loading dataset_v6...")
df_v6 = pd.read_csv("dataset_v6/labels.csv")
print(f"Loaded {len(df_v6)} annotations")
print(f"Omega range:  {df_v6['omega'].min():.2f} to {df_v6['omega'].max():.2f}  (must be within -30 to +30)")
print(f"Radius range: {df_v6['radius'].min():.3f} to {df_v6['radius'].max():.3f}")

tr6, va6 = get_splits(df_v6)

print("\nSingle-Frame CNN v6...")
p,t = train_and_eval(PhysicsCNN(3,2).cuda(), SingleFrameDS(tr6,"dataset_v6/images"), SingleFrameDS(va6,"dataset_v6/images"), name="Single-v6")
v6_single = (float(np.abs(p[:,0]-t[:,0]).mean()), float(np.abs(p[:,1]-t[:,1]).mean()))

print("\nTwo-Frame CNN v6...")
p,t = train_and_eval(PhysicsCNN(6,2).cuda(), TwoFrameDS(tr6,"dataset_v6/images"), TwoFrameDS(va6,"dataset_v6/images"), name="Two-v6")
v6_two = (float(np.abs(p[:,0]-t[:,0]).mean()), float(np.abs(p[:,1]-t[:,1]).mean()))

print("\nEvent CNN v6...")
p,t = train_and_eval(PhysicsCNN(2,2).cuda(), EventDS(tr6,"dataset_v6/images"), EventDS(va6,"dataset_v6/images"), name="Event-v6")
v6_event = (float(np.abs(p[:,0]-t[:,0]).mean()), float(np.abs(p[:,1]-t[:,1]).mean()))

print("\n5-Frame Sequence CNN v6 (7 outputs: vx vy gravity restitution friction omega radius)...")
p,t = train_and_eval(PhysicsCNN(10,7).cuda(), SeqEventV6DS(tr6,"dataset_v6/images"), SeqEventV6DS(va6,"dataset_v6/images"), name="Seq-v6")
v6_seq = {
    'vx':  float(np.abs(p[:,0]-t[:,0]).mean()),
    'vy':  float(np.abs(p[:,1]-t[:,1]).mean()),
    'g':   float(np.abs(p[:,2]-t[:,2]).mean()),
    'r':   float(np.abs(p[:,3]-t[:,3]).mean()),
    'f':   float(np.abs(p[:,4]-t[:,4]).mean()),
    'om':  float(np.abs(p[:,5]-t[:,5]).mean()),
    'rad': float(np.abs(p[:,6]-t[:,6]).mean()),
}

print("V6 RESULTS")
print(f"  Single-Frame:  vx={v6_single[0]:.4f}  vy={v6_single[1]:.4f}")
print(f"  Two-Frame:     vx={v6_two[0]:.4f}  vy={v6_two[1]:.4f}")
print(f"  Event CNN:     vx={v6_event[0]:.4f}  vy={v6_event[1]:.4f}")
print(f"  5-Frame Seq:   vx={v6_seq['vx']:.4f}  vy={v6_seq['vy']:.4f}  g={v6_seq['g']:.4f}  rest={v6_seq['r']:.4f}  fric={v6_seq['f']:.4f}  omega={v6_seq['om']:.4f}  radius={v6_seq['rad']:.4f}")

Dataset classes for v4/v5 ready
Loading dataset_v6...
Loaded 120000 annotations
Omega range:  -30.00 to 30.00  (must be within -30 to +30)
Radius range: 0.030 to 0.080

Single-Frame CNN v6...
  Single-v6 Epoch 5  val_loss=0.9929
  Single-v6 Epoch 10  val_loss=0.9941
  Single-v6 Epoch 15  val_loss=0.9919
  Single-v6 Epoch 20  val_loss=0.9922

Two-Frame CNN v6...
  Two-v6 Epoch 5  val_loss=1.7049
  Two-v6 Epoch 10  val_loss=1.0004
  Two-v6 Epoch 15  val_loss=0.8856
  Two-v6 Epoch 20  val_loss=0.9059

Event CNN v6...
  Event-v6 Epoch 5  val_loss=0.8903
  Event-v6 Epoch 10  val_loss=0.8822
  Event-v6 Epoch 15  val_loss=0.8916
  Event-v6 Epoch 20  val_loss=0.8908

5-Frame Sequence CNN v6 (7 outputs: vx vy gravity restitution friction omega radius)...
  Seq-v6 Epoch 5  val_loss=0.8749
  Seq-v6 Epoch 10  val_loss=0.8942
  Seq-v6 Epoch 15  val_loss=0.8857
  Seq-v6 Epoch 20  val_loss=0.8947
V6 RESULTS
  Single-Frame:  vx=0.6347  vy=0.5719
  Two-Frame:     vx=0.5875  vy=0.6090
  Event CNN:     v

In [ ]:
import warp as wp
import numpy as np
import os, cv2
import pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

wp.init()

class PhysicsCNN(nn.Module):
    def __init__(self, in_ch, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,   64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,  128,  3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(128*8*8, 256), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, out_dim)
        )
    def forward(self, x): return self.fc(self.conv(x))

def train_and_eval(model, train_ds, val_ds, epochs=20, name=""):
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            criterion(model(x.cuda()), y.cuda()).backward()
            optimizer.step()
        model.eval()
        vl = np.mean([criterion(model(x.cuda()), y.cuda()).item() for x, y in val_loader])
        scheduler.step(vl)
        if (epoch+1) % 5 == 0:
            print(f"  {name} Epoch {epoch+1}  val_loss={vl:.4f}")
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, y in val_loader:
            all_p.append(model(x.cuda()).cpu())
            all_t.append(y)
    return torch.cat(all_p).numpy(), torch.cat(all_t).numpy()

def get_splits(df):
    n  = int(0.8 * df['step'].nunique())
    ts = df['step'].unique()[:n]
    vs = df['step'].unique()[n:]
    return df[df['step'].isin(ts)].reset_index(drop=True), df[df['step'].isin(vs)].reset_index(drop=True)

class SingleFrameDS(Dataset):
    def __init__(self,df,img_dir):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir
        self.t=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.5,.5,.5],[.5,.5,.5])])
        self.vx_mean,self.vx_std=df['vx'].mean(),df['vx'].std()+1e-8
        self.vy_mean,self.vy_std=df['vy'].mean(),df['vy'].std()+1e-8
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        img=self.t(Image.open(f"{self.img_dir}/env{int(row.env_id)}_step{int(row.step)}.png").convert("RGB"))
        return img,torch.tensor([(row.vx-self.vx_mean)/self.vx_std,(row.vy-self.vy_mean)/self.vy_std],dtype=torch.float32)

class TwoFrameDS(Dataset):
    def __init__(self,df,img_dir):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir
        self.valid=[i for i in range(len(df)) if int(df.iloc[i].step)>0]
        self.t=transforms.Compose([transforms.ToTensor(),transforms.Normalize([.5,.5,.5],[.5,.5,.5])])
        self.vx_mean,self.vx_std=df['vx'].mean(),df['vx'].std()+1e-8
        self.vy_mean,self.vy_std=df['vy'].mean(),df['vy'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self,idx):
        row=self.df.iloc[self.valid[idx]]; step,env=int(row.step),int(row.env_id)
        curr=self.t(Image.open(f"{self.img_dir}/env{env}_step{step}.png").convert("RGB"))
        prev=self.t(Image.open(f"{self.img_dir}/env{env}_step{step-1}.png").convert("RGB"))
        return torch.cat([curr,prev],dim=0),torch.tensor([(row.vx-self.vx_mean)/self.vx_std,(row.vy-self.vy_mean)/self.vy_std],dtype=torch.float32)

class EventDS(Dataset):
    def __init__(self,df,img_dir,threshold=10):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir; self.threshold=threshold
        self.valid=[i for i in range(len(df)) if int(df.iloc[i].step)>0]
        self.vx_mean,self.vx_std=df['vx'].mean(),df['vx'].std()+1e-8
        self.vy_mean,self.vy_std=df['vy'].mean(),df['vy'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self,idx):
        row=self.df.iloc[self.valid[idx]]; step,env=int(row.step),int(row.env_id)
        curr=np.array(Image.open(f"{self.img_dir}/env{env}_step{step}.png")).astype(float)
        prev=np.array(Image.open(f"{self.img_dir}/env{env}_step{step-1}.png")).astype(float)
        diff=curr.mean(2)-prev.mean(2)
        event=torch.tensor(np.stack([(diff>self.threshold).astype(np.float32),(diff<-self.threshold).astype(np.float32)]))
        return event,torch.tensor([(row.vx-self.vx_mean)/self.vx_std,(row.vy-self.vy_mean)/self.vy_std],dtype=torch.float32)

class SeqEventDS(Dataset):
    def __init__(self,df,img_dir,seq_len=5,threshold=10):
        self.df=df.reset_index(drop=True); self.img_dir=img_dir
        self.seq_len=seq_len; self.threshold=threshold
        self.valid=[i for i in range(len(df)) if int(df.iloc[i].step)>=seq_len]
        self.vx_mean,self.vx_std=df['vx'].mean(),df['vx'].std()+1e-8
        self.vy_mean,self.vy_std=df['vy'].mean(),df['vy'].std()+1e-8
        self.g_mean,self.g_std=df['gravity'].mean(),df['gravity'].std()+1e-8
        self.r_mean,self.r_std=df['restitution'].mean(),df['restitution'].std()+1e-8
        self.f_mean,self.f_std=df['friction'].mean(),df['friction'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self,idx):
        row=self.df.iloc[self.valid[idx]]; step,env=int(row.step),int(row.env_id)
        frames=[]
        for s in range(step-self.seq_len+1,step+1):
            img=np.array(Image.open(f"{self.img_dir}/env{env}_step{s}.png")).astype(float)
            if s>step-self.seq_len+1:
                prev=np.array(Image.open(f"{self.img_dir}/env{env}_step{s-1}.png")).astype(float)
                diff=img.mean(2)-prev.mean(2)
                frames+=[(diff>self.threshold).astype(np.float32),(diff<-self.threshold).astype(np.float32)]
            else:
                frames+=[np.zeros((64,64),dtype=np.float32),np.zeros((64,64),dtype=np.float32)]
        return torch.tensor(np.stack(frames)),torch.tensor([
            (row.vx-self.vx_mean)/self.vx_std,(row.vy-self.vy_mean)/self.vy_std,
            (row.gravity-self.g_mean)/self.g_std,(row.restitution-self.r_mean)/self.r_std,
            (row.friction-self.f_mean)/self.f_std],dtype=torch.float32)

print("All restored - ready to train")

All restored - ready to train


In [ ]:
# Train all models on v4 and v5
df_v4 = pd.read_csv("dataset_v4/labels.csv")
df_v5 = pd.read_csv("dataset_v5/labels.csv")

results = {}

for ds_name, df in [("v4", df_v4), ("v5", df_v5)]:
    img_dir = f"dataset_{ds_name}/images"
    print(f"TRAINING ON DATASET {ds_name.upper()}")
    tr, va = get_splits(df)

    print(f"\nSingle-Frame CNN {ds_name}...")
    p,t = train_and_eval(PhysicsCNN(3,2).cuda(), SingleFrameDS(tr,img_dir), SingleFrameDS(va,img_dir), name=f"Single-{ds_name}")
    single = (float(np.abs(p[:,0]-t[:,0]).mean()), float(np.abs(p[:,1]-t[:,1]).mean()))

    print(f"\nTwo-Frame CNN {ds_name}...")
    p,t = train_and_eval(PhysicsCNN(6,2).cuda(), TwoFrameDS(tr,img_dir), TwoFrameDS(va,img_dir), name=f"Two-{ds_name}")
    two = (float(np.abs(p[:,0]-t[:,0]).mean()), float(np.abs(p[:,1]-t[:,1]).mean()))

    print(f"\nEvent CNN {ds_name}...")
    p,t = train_and_eval(PhysicsCNN(2,2).cuda(), EventDS(tr,img_dir), EventDS(va,img_dir), name=f"Event-{ds_name}")
    event = (float(np.abs(p[:,0]-t[:,0]).mean()), float(np.abs(p[:,1]-t[:,1]).mean()))

    print(f"\n5-Frame Sequence CNN {ds_name}...")
    p,t = train_and_eval(PhysicsCNN(10,5).cuda(), SeqEventDS(tr,img_dir), SeqEventDS(va,img_dir), name=f"Seq-{ds_name}")
    seq = {'vx':float(np.abs(p[:,0]-t[:,0]).mean()), 'vy':float(np.abs(p[:,1]-t[:,1]).mean()),
           'g':float(np.abs(p[:,2]-t[:,2]).mean()), 'r':float(np.abs(p[:,3]-t[:,3]).mean()),
           'f':float(np.abs(p[:,4]-t[:,4]).mean())}

    results[ds_name] = {'single':single, 'two':two, 'event':event, 'seq':seq}

print("V4 AND V5 RESULTS SUMMARY")
for ds_name in ["v4","v5"]:
    r = results[ds_name]
    print(f"\nDataset {ds_name}:")
    print(f"  Single-Frame:  vx={r['single'][0]:.4f}  vy={r['single'][1]:.4f}")
    print(f"  Two-Frame:     vx={r['two'][0]:.4f}  vy={r['two'][1]:.4f}")
    print(f"  Event CNN:     vx={r['event'][0]:.4f}  vy={r['event'][1]:.4f}")
    print(f"  5-Frame Seq:   vx={r['seq']['vx']:.4f}  vy={r['seq']['vy']:.4f}  g={r['seq']['g']:.4f}  rest={r['seq']['r']:.4f}  fric={r['seq']['f']:.4f}")

TRAINING ON DATASET V4

Single-Frame CNN v4...
  Single-v4 Epoch 5  val_loss=1.0021
  Single-v4 Epoch 10  val_loss=1.0170
  Single-v4 Epoch 15  val_loss=1.0078
  Single-v4 Epoch 20  val_loss=1.0063

Two-Frame CNN v4...
  Two-v4 Epoch 5  val_loss=1.1396
  Two-v4 Epoch 10  val_loss=1.0386
  Two-v4 Epoch 15  val_loss=0.9293
  Two-v4 Epoch 20  val_loss=0.9441

Event CNN v4...
  Event-v4 Epoch 5  val_loss=0.9202
  Event-v4 Epoch 10  val_loss=0.9266
  Event-v4 Epoch 15  val_loss=0.9203
  Event-v4 Epoch 20  val_loss=0.9205

5-Frame Sequence CNN v4...
  Seq-v4 Epoch 5  val_loss=0.8139
  Seq-v4 Epoch 10  val_loss=0.8115
  Seq-v4 Epoch 15  val_loss=0.7958
  Seq-v4 Epoch 20  val_loss=0.8053
TRAINING ON DATASET V5

Single-Frame CNN v5...
  Single-v5 Epoch 5  val_loss=0.9959
  Single-v5 Epoch 10  val_loss=0.9931
  Single-v5 Epoch 15  val_loss=0.9937
  Single-v5 Epoch 20  val_loss=0.9921

Two-Frame CNN v5...
  Two-v5 Epoch 5  val_loss=1.1842
  Two-v5 Epoch 10  val_loss=0.9282
  Two-v5 Epoch 15  val_

In [ ]:
# v6 training + noise experiment + fusion + % error table
class EventDS(Dataset):
    def __init__(self, df, img_dir, threshold=10):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.threshold = threshold
        self.valid = [i for i in range(len(df)) if int(df.iloc[i].step) > 0]
        self.vx_mean, self.vx_std = df['vx'].mean(), df['vx'].std()+1e-8
        self.vy_mean, self.vy_std = df['vy'].mean(), df['vy'].std()+1e-8
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        curr = np.array(Image.open(f"{self.img_dir}/env{env}_step{step}.png")).astype(float)
        prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{step-1}.png")).astype(float)
        diff = curr.mean(2) - prev.mean(2)
        event = torch.tensor(np.stack([(diff>self.threshold).astype(np.float32),
                                        (diff<-self.threshold).astype(np.float32)]))
        return event, torch.tensor([(row.vx-self.vx_mean)/self.vx_std,
                                     (row.vy-self.vy_mean)/self.vy_std], dtype=torch.float32)

noise_levels = [0.0, 0.05, 0.10, 0.20]
noise_results = {}

for ds_name, df_noise in [("v4", df_v4), ("v5", df_v5)]:
    img_dir = f"dataset_{ds_name}/images"
    print(f"NOISE EXPERIMENT ON {ds_name.upper()}")
    tr, va = get_splits(df_noise)
    noise_results[ds_name] = {}
    vx_m = tr['vx'].mean(); vx_s = tr['vx'].std()+1e-8
    vy_m = tr['vy'].mean(); vy_s = tr['vy'].std()+1e-8
    thr  = 10

    for noise_std in noise_levels:
        class NoisyEventDS(Dataset):
            def __init__(self, df, img_dir, ns):
                self.df      = df.reset_index(drop=True)
                self.img_dir = img_dir
                self.ns      = ns
                self.valid   = [i for i in range(len(df)) if int(df.iloc[i].step) > 0]
            def __len__(self): return len(self.valid)
            def __getitem__(self, idx):
                row  = self.df.iloc[self.valid[idx]]
                step, env = int(row.step), int(row.env_id)
                curr = np.array(Image.open(f"{self.img_dir}/env{env}_step{step}.png")).astype(float)
                prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{max(step-1,0)}.png")).astype(float)
                diff = curr.mean(2) - prev.mean(2)
                pos_ev = (diff >  thr).astype(np.float32)
                neg_ev = (diff < -thr).astype(np.float32)
                event  = np.stack([pos_ev, neg_ev])
                if self.ns > 0:
                    event = np.clip(event + np.random.normal(0, self.ns, event.shape).astype(np.float32), 0, 1)
                return torch.tensor(event), torch.tensor(
                    [(row.vx-vx_m)/vx_s, (row.vy-vy_m)/vy_s], dtype=torch.float32)

        print(f"\n  noise_std={noise_std}")
        p, t = train_and_eval(PhysicsCNN(2,2).cuda(),
                              NoisyEventDS(tr, img_dir, noise_std),
                              NoisyEventDS(va, img_dir, noise_std),
                              name=f"Noise{noise_std}-{ds_name}")
        vx_mae = float(np.abs(p[:,0]-t[:,0]).mean())
        vy_mae = float(np.abs(p[:,1]-t[:,1]).mean())
        noise_results[ds_name][noise_std] = (vx_mae, vy_mae)
        print(f"  >> vx MAE={vx_mae:.4f}  vy MAE={vy_mae:.4f}")

print("NOISE RESULTS SUMMARY")
for ds in ["v4","v5"]:
    print(f"\nDataset {ds}:")
    print(f"  {'Noise':<14} {'vx MAE':<10} {'vy MAE'}")
    for ns,(vx,vy) in noise_results[ds].items():
        label = "0.00 (clean)" if ns==0.0 else f"{ns:.2f}"
        print(f"  {label:<14} {vx:<10.4f} {vy:.4f}")

#  RGB+Event fusion
print("RGB+EVENT FUSION EXPERIMENT")

tr4, va4 = get_splits(df_v4)
vx_m4 = df_v4['vx'].mean(); vx_s4 = df_v4['vx'].std()+1e-8
vy_m4 = df_v4['vy'].mean(); vy_s4 = df_v4['vy'].std()+1e-8

class RGBEventDS(Dataset):
    def __init__(self, df, img_dir, threshold=10):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.threshold = threshold
        self.valid     = [i for i in range(len(df)) if int(df.iloc[i].step) > 0]
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row  = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        curr = np.array(Image.open(f"{self.img_dir}/env{env}_step{step}.png").convert('RGB')).astype(np.float32)/255.0
        prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{max(step-1,0)}.png").convert('RGB')).astype(np.float32)/255.0
        rgb   = torch.tensor(curr.transpose(2,0,1))
        diff  = curr.mean(2) - prev.mean(2)
        thresh = self.threshold / 255.0
        event = torch.tensor(np.stack([(diff>thresh).astype(np.float32),
                                        (diff<-thresh).astype(np.float32)]))
        return torch.cat([rgb,event],dim=0), torch.tensor(
            [(row.vx-vx_m4)/vx_s4, (row.vy-vy_m4)/vy_s4], dtype=torch.float32)

print("\nTraining RGB+Event Fusion (5ch) on v4...")
p, t = train_and_eval(PhysicsCNN(5,2).cuda(),
                      RGBEventDS(tr4,"dataset_v4/images"),
                      RGBEventDS(va4,"dataset_v4/images"),
                      name="Fusion-v4")
fusion_vx = float(np.abs(p[:,0]-t[:,0]).mean())
fusion_vy = float(np.abs(p[:,1]-t[:,1]).mean())

print("FUSION RESULTS SUMMARY")
print(f"{'Model':<25} {'vx MAE':<12} {'vy MAE'}")
print(f"{'Event only (2ch)':<25} {results['v4']['event'][0]:<12.4f} {results['v4']['event'][1]:.4f}")
print(f"{'RGB+Event (5ch)':<25} {fusion_vx:<12.4f} {fusion_vy:.4f}")

print("PERCENTAGE ERROR TABLE")
seq_v4 = results['v4']['seq']
param_info = {
    'vx':          {'mae': seq_v4['vx'], 'range': 3.0,  'unit': 'sim units/s'},
    'vy':          {'mae': seq_v4['vy'], 'range': 3.0,  'unit': 'sim units/s'},
    'gravity':     {'mae': seq_v4['g'],  'range': 7.0,  'unit': 'sim units/s²'},
    'restitution': {'mae': seq_v4['r'],  'range': 0.25, 'unit': 'dimensionless'},
    'friction':    {'mae': seq_v4['f'],  'range': 1.0,  'unit': 'dimensionless'},
}
print(f"\n{'Parameter':<15} {'MAE':<8} {'Unit':<20} {'Range':<8} {'% Error'}")
for param, info in param_info.items():
    pct = (info['mae'] / info['range']) * 100
    print(f"{param:<15} {info['mae']:<8.3f} {info['unit']:<20} {info['range']:<8.2f} {pct:.1f}%")
print("\nNote: MAE in normalised simulator units, domain [0,1]x[0,1]")
print("% Error = MAE / parameter range x 100")

NOISE EXPERIMENT ON V4

  noise_std=0.0
  Noise0.0-v4 Epoch 5  val_loss=0.1177
  Noise0.0-v4 Epoch 10  val_loss=0.1165
  Noise0.0-v4 Epoch 15  val_loss=0.1162
  Noise0.0-v4 Epoch 20  val_loss=0.1162
  >> vx MAE=0.0644  vy MAE=0.2135

  noise_std=0.05
  Noise0.05-v4 Epoch 5  val_loss=0.1183
  Noise0.05-v4 Epoch 10  val_loss=0.1176
  Noise0.05-v4 Epoch 15  val_loss=0.1151
  Noise0.05-v4 Epoch 20  val_loss=0.1158
  >> vx MAE=0.0665  vy MAE=0.2122

  noise_std=0.1
  Noise0.1-v4 Epoch 5  val_loss=0.1160
  Noise0.1-v4 Epoch 10  val_loss=0.1150
  Noise0.1-v4 Epoch 15  val_loss=0.1147
  Noise0.1-v4 Epoch 20  val_loss=0.1146
  >> vx MAE=0.0628  vy MAE=0.1930

  noise_std=0.2
  Noise0.2-v4 Epoch 5  val_loss=0.1220
  Noise0.2-v4 Epoch 10  val_loss=0.1194
  Noise0.2-v4 Epoch 15  val_loss=0.1150
  Noise0.2-v4 Epoch 20  val_loss=0.1158
  >> vx MAE=0.0652  vy MAE=0.1872
NOISE EXPERIMENT ON V5

  noise_std=0.0
  Noise0.0-v5 Epoch 5  val_loss=0.1170
  Noise0.0-v5 Epoch 10  val_loss=0.1170
  Noise0.0-v5

In [ ]:
import json
save_data = {
    'v4_results': results,
    'v6_single': v6_single,
    'v6_two': v6_two,
    'v6_event': v6_event,
    'v6_seq': v6_seq,
    'noise_v4': {
        0.0:  (0.0644, 0.2135),
        0.05: (0.0665, 0.2122),
        0.1:  (0.0628, 0.1930),
        0.2:  (0.0652, 0.1872)
    }
}
with open('/kaggle/working/saved_results.json', 'w') as f:
    json.dump(save_data, f)
print("SAVED")
print(save_data)

In [4]:
import warp as wp
import numpy as np
import os, cv2
import pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
wp.init()
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

Torch: 2.10.0+cpu | CUDA: False


In [5]:
class PhysicsCNN(nn.Module):
    def __init__(self, in_ch, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, 32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,   64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,  128,  3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(), nn.Linear(128*8*8, 256), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, out_dim)
        )
    def forward(self, x): return self.fc(self.conv(x))

def train_and_eval(model, train_ds, val_ds, epochs=20, name=''):
    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=0)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            criterion(model(x.cuda()), y.cuda()).backward()
            optimizer.step()
        model.eval()
        vl = np.mean([criterion(model(x.cuda()), y.cuda()).item() for x, y in val_loader])
        scheduler.step(vl)
        if (epoch+1) % 5 == 0:
            print(f"  {name} Epoch {epoch+1}  val_loss={vl:.4f}")
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, y in val_loader:
            all_p.append(model(x.cuda()).cpu())
            all_t.append(y)
    return torch.cat(all_p).numpy(), torch.cat(all_t).numpy()

def get_splits(df):
    n  = int(0.8 * df['step'].nunique())
    ts = df['step'].unique()[:n]
    vs = df['step'].unique()[n:]
    return df[df['step'].isin(ts)].reset_index(drop=True), df[df['step'].isin(vs)].reset_index(drop=True)

print("Model and helpers ready")

Model and helpers ready


In [8]:
df_v4 = pd.read_csv("dataset_v4/labels.csv")
tr4, va4 = get_splits(df_v4)
vx_m4 = df_v4['vx'].mean(); vx_s4 = df_v4['vx'].std()+1e-8
vy_m4 = df_v4['vy'].mean(); vy_s4 = df_v4['vy'].std()+1e-8

class RGBEventDS(Dataset):
    def __init__(self, df, img_dir, threshold=10):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir; self.threshold = threshold
        self.valid = [i for i in range(len(df)) if int(df.iloc[i].step) > 0]
    def __len__(self): return len(self.valid)
    def __getitem__(self, idx):
        row = self.df.iloc[self.valid[idx]]
        step, env = int(row.step), int(row.env_id)
        curr = np.array(Image.open(f"{self.img_dir}/env{env}_step{step}.png").convert('RGB')).astype(np.float32)/255.0
        prev = np.array(Image.open(f"{self.img_dir}/env{env}_step{max(step-1,0)}.png").convert('RGB')).astype(np.float32)/255.0
        rgb = torch.tensor(curr.transpose(2,0,1))
        diff = curr.mean(2) - prev.mean(2)
        thresh = self.threshold / 255.0
        event = torch.tensor(np.stack([(diff>thresh).astype(np.float32),(diff<-thresh).astype(np.float32)]))
        return torch.cat([rgb,event],dim=0), torch.tensor(
            [(row.vx-vx_m4)/vx_s4, (row.vy-vy_m4)/vy_s4], dtype=torch.float32)

print("Training RGB+Event Fusion (5ch) on v4...")
p, t = train_and_eval(PhysicsCNN(5,2).cuda(),
                      RGBEventDS(tr4,"dataset_v4/images"),
                      RGBEventDS(va4,"dataset_v4/images"),
                      name="Fusion-v4")
fusion_vx = float(np.abs(p[:,0]-t[:,0]).mean())
fusion_vy = float(np.abs(p[:,1]-t[:,1]).mean())
print(f"FUSION RESULT: vx={fusion_vx:.4f}  vy={fusion_vy:.4f}")
# Event-only baseline for comparison: vx=0.2090  vy=0.4590

Training RGB+Event Fusion (5ch) on v4...
  Fusion-v4 Epoch 5  val_loss=0.1745
  Fusion-v4 Epoch 10  val_loss=0.1775
  Fusion-v4 Epoch 15  val_loss=0.1762
  Fusion-v4 Epoch 20  val_loss=0.1775
FUSION RESULT: vx=0.1054  vy=0.2456
